In [1]:
# Cell 1: Parse the PDF using Databricks ai_parse_document
# Reads the 1603-R2 TIL PDF from the volume and runs the Databricks built-in parser.
# Result is a dataframe with a nested 'parsed_content' struct.

from pyspark.sql.functions import expr

docs_df = spark.read.format('binaryFile').load('/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/TILS_new/TIL 1603-R2 - R0 EROSION AND WATER INGESTION RECOMMENDATIONS.pdf')
parsed_df = docs_df.withColumn(
    'parsed_content',
    expr(
        "ai_parse_document(content, map('version', '2.0', 'imageOutputPath', '/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/parsed_images/'))"
    )
)
parsed_df = parsed_df.drop('content')
print('Parse complete. Schema:')
parsed_df.printSchema()


ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Cell 2: Inspect parsed_content.metadata
# Shows parser run info: version, page count, parse status, timing.
# Useful to confirm the parser ran cleanly and which version was used.

from pyspark.sql.functions import col

print('--- parsed_content.metadata ---')
parsed_df.select('parsed_content.metadata').show(truncate=False)


In [ ]:
# Cell 3: Inspect parsed_content.document.pages
# Each row is one page. Shows page number, raw text content, and layout info.
# This is where you check reading order and section completeness.

from pyspark.sql.functions import col, explode

pages_df = parsed_df.select(
    explode(col('parsed_content.document.pages')).alias('page')
).select(
    col('page.pageNumber').alias('page_number'),
    col('page.text').alias('text')
)

print(f'Total pages: {pages_df.count()}')
print()

for row in pages_df.orderBy('page_number').collect():
    print(f'=== Page {row.page_number} ===')
    print(row.text or '(no text)')
    print()


In [ ]:
# Cell 4: Inspect parsed_content.document.elements
# Each row is one extracted element: TEXT, TABLE, or IMAGE.
# Shows element type distribution and lets you inspect each type separately.

from pyspark.sql.functions import col, explode

elements_df = parsed_df.select(
    explode(col('parsed_content.document.elements')).alias('el')
).select(
    col('el.type').alias('type'),
    col('el.pageNumber').alias('page_number'),
    col('el.text').alias('text')
)

print('--- Element type counts ---')
elements_df.groupBy('type').count().orderBy('type').show()


In [ ]:
# Cell 5: Inspect TABLE elements only
# Shows table content per page. Compare these against DS extracted tables in:
#   TILs/input/ds-code-and-results/1603-R2/1603-R2/extracted_document.md
# Key tables to check: compliance/timing table (page 1), wet inlet interval table (page 5).

from pyspark.sql.functions import col, explode

tables_df = elements_df.filter(col('type') == 'TABLE').orderBy('page_number')

print(f'Total TABLE elements: {tables_df.count()}')
print()

for row in tables_df.collect():
    print(f'--- Table on page {row.page_number} ---')
    print(row.text or '(empty)')
    print()


In [ ]:
# Cell 6: Inspect IMAGE elements only
# Shows image descriptions or alt text per page.
# Compare against DS image descriptions in extracted_document.md to assess signal vs noise.

images_df = elements_df.filter(col('type') == 'IMAGE').orderBy('page_number')

print(f'Total IMAGE elements: {images_df.count()}')
print()

for row in images_df.collect():
    print(f'--- Image on page {row.page_number} ---')
    print(row.text or '(no description)')
    print()
